# Análise do Inventário Florestal

Exploração dos 31 CSVs do dataset **Forest Inventory Brazil 2007 (ORNL DAAC)**.

**Fonte:** `data/raw/inventory/` — ver README dessa pasta para detalhes.

**Roteiro:**
1. Carregamento e padronização dos CSVs (colunas variam entre sites)
2. Visão geral (árvores, sites, plots)
3. Distribuição de DAP (diâmetro à altura do peito)
4. Riqueza de espécies por site
5. Mortalidade entre campanhas
6. Cobertura temporal
7. Estimativa de biomassa via alometria (Brown 1997)

## 0. Setup

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

ROOT = Path('.').resolve()
for _ in range(5):
    if (ROOT / 'data').exists(): break
    ROOT = ROOT.parent

INV_DIR = next(p for p in (ROOT / 'data/raw/inventory').iterdir() if p.is_dir())
print('Dataset:', INV_DIR.name)

## 1. Carregamento e padronização

Os CSVs têm nomes de colunas inconsistentes entre sites. Normalizamos para um esquema comum.

In [ ]:
def normalize_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Normaliza nomes de colunas para um esquema comum."""
    col_map = {}
    for c in df.columns:
        cl = c.lower().replace('.', '_').replace(' ', '_').strip()
        col_map[c] = cl
    df = df.rename(columns=col_map)

    # DBH — pega a primeira coluna que contenha 'dbh'
    dbh_cols = sorted([c for c in df.columns if 'dbh' in c])
    if dbh_cols and 'dbh' not in df.columns:
        df['dbh'] = pd.to_numeric(df[dbh_cols[0]], errors='coerce')

    # Coordenadas
    for alias, target in [('utm_easting','easting'), ('utmeasting','easting'),
                          ('utm_northing','northing'), ('utmnorthing','northing')]:
        if alias in df.columns and target not in df.columns:
            df[target] = df[alias]

    # Nome científico
    for alias in ('scientific_name', 'scienfic_name_', 'scienfic_name'):
        if alias in df.columns and 'scientific_name' not in df.columns:
            df['scientific_name'] = df[alias]

    # Mortalidade
    dead_cols = [c for c in df.columns if 'dead' in c]
    if dead_cols and 'dead' not in df.columns:
        df['dead'] = df[dead_cols[0]]

    return df


csvs = sorted(INV_DIR.glob('*nventory*.csv'))
frames = []
for csv in csvs:
    try:
        raw = pd.read_csv(csv, encoding='latin-1', low_memory=False)
        raw['source_file'] = csv.name
        frames.append(normalize_cols(raw))
    except Exception as e:
        print(f'Erro em {csv.name}: {e}')

df = pd.concat(frames, ignore_index=True)
print(f'{len(csvs)} CSVs carregados → {len(df):,} linhas')
print(f'Colunas comuns disponíveis: {[c for c in ["dbh","scientific_name","dead","easting","northing"] if c in df.columns]}')

## 2. Visão geral

In [ ]:
# Extrai site e plot das colunas disponíveis
site_col = next((c for c in df.columns if c in ('area','site','area_id')), None)
plot_col = next((c for c in df.columns if 'plot' in c), None)

print('=== Visão geral ===')
print(f'Total de registros (árvores):  {len(df):,}')
if site_col:
    print(f'Sites únicos:                  {df[site_col].nunique()}')
if plot_col:
    print(f'Plots únicos:                  {df[plot_col].nunique()}')
if 'scientific_name' in df.columns:
    print(f'Espécies únicas:               {df["scientific_name"].nunique()}')
if 'dbh' in df.columns:
    valid_dbh = df['dbh'].dropna()
    print(f'Árvores com DAP válido:        {len(valid_dbh):,}')
    print(f'DAP mediano:                   {valid_dbh.median():.1f} cm')
    print(f'DAP máximo:                    {valid_dbh.max():.1f} cm')

## 3. Distribuição de DAP

In [ ]:
if 'dbh' not in df.columns:
    print('Coluna DBH não encontrada — verifique o mapeamento.')
else:
    dbh = df['dbh'].dropna()
    dbh = dbh[(dbh > 0) & (dbh < 500)]  # remove outliers extremos

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histograma geral
    axes[0].hist(dbh, bins=60, color='forestgreen', edgecolor='white', linewidth=0.3)
    for pct, c in [(50,'red'),(90,'darkorange'),(99,'purple')]:
        val = dbh.quantile(pct/100)
        axes[0].axvline(val, linestyle='--', color=c, label=f'P{pct}={val:.0f}cm')
    axes[0].set_xlabel('DAP (cm)'); axes[0].set_ylabel('Nº de árvores')
    axes[0].set_title('Distribuição de DAP — todos os sites')
    axes[0].legend(fontsize=8)

    # Classes diamétricas (distribuição em J-invertido esperada)
    bins = [10, 20, 30, 40, 50, 60, 80, 100, 150, 200, 500]
    counts, edges = np.histogram(dbh, bins=bins)
    labels = [f'{a}-{b}' for a, b in zip(bins[:-1], bins[1:])]
    axes[1].bar(range(len(counts)), counts, color='steelblue', edgecolor='white')
    axes[1].set_xticks(range(len(counts)))
    axes[1].set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
    axes[1].set_xlabel('Classe diamétrica (cm)'); axes[1].set_ylabel('Nº de árvores')
    axes[1].set_title('Classes diamétricas')

    plt.tight_layout()
    plt.show()
    print(f'Árvores com DAP ≥ 100 cm (emergentes): {(dbh >= 100).sum()}')

## 4. Riqueza de espécies

In [ ]:
if 'scientific_name' not in df.columns:
    print('Coluna scientific_name não encontrada.')
else:
    sp = df['scientific_name'].dropna().str.strip()
    sp = sp[sp != '']

    # Top 20 espécies mais frequentes
    top20 = sp.value_counts().head(20)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(top20.index[::-1], top20.values[::-1], color='steelblue')
    axes[0].set_xlabel('Nº de registros')
    axes[0].set_title('Top 20 espécies mais frequentes')
    axes[0].tick_params(axis='y', labelsize=8)

    # Riqueza por site
    if site_col:
        richness = df.groupby(site_col)['scientific_name'].nunique().sort_values(ascending=False)
        axes[1].bar(richness.index, richness.values, color='darkorange')
        axes[1].set_xlabel('Site'); axes[1].set_ylabel('Nº de espécies')
        axes[1].set_title('Riqueza de espécies por site')
        axes[1].tick_params(axis='x', rotation=45, labelsize=7)

    plt.tight_layout()
    plt.show()
    print(f'Total de espécies únicas: {sp.nunique()}')
    print(f'Gêneros únicos: {sp.str.split().str[0].nunique()}')

## 5. Mortalidade entre campanhas

In [ ]:
dead_candidates = [c for c in df.columns if 'dead' in c.lower()]
print('Colunas de mortalidade encontradas:', dead_candidates)

if dead_candidates:
    dead_col = dead_candidates[0]
    dead = df[dead_col].map({True: True, False: False, 'True': True, 'False': False,
                              1: True, 0: False, '1': True, '0': False})
    n_dead = dead.sum()
    n_total = dead.notna().sum()
    print(f'\nMortalidade: {n_dead} de {n_total} árvores ({100*n_dead/n_total:.1f}%)')

    if site_col:
        mort_by_site = df.groupby(site_col).apply(
            lambda g: pd.Series({
                'n_trees': len(g),
                'n_dead': g[dead_col].map({True:1,False:0,'True':1,'False':0}).sum(),
            })
        )
        mort_by_site['pct_dead'] = 100 * mort_by_site['n_dead'] / mort_by_site['n_trees']

        fig, ax = plt.subplots(figsize=(12, 4))
        ax.bar(mort_by_site.index, mort_by_site['pct_dead'], color='tomato')
        ax.set_xlabel('Site'); ax.set_ylabel('Mortalidade (%)')
        ax.set_title('Taxa de mortalidade por site')
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.axhline(mort_by_site['pct_dead'].mean(), linestyle='--', color='black',
                   label=f'Média={mort_by_site["pct_dead"].mean():.1f}%')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('Nenhuma coluna de mortalidade encontrada.')

## 6. Estimativa de biomassa via alometria

Usamos a equação de **Brown (1997)** para floresta tropical úmida, que depende apenas do DAP:

$$AGB = \exp\left(-2.289 + 2.649 \cdot \ln(D) - 0.021 \cdot (\ln(D))^2\right)$$

Onde $D$ é o DAP em cm e $AGB$ é a biomassa acima do solo em kg por árvore.

In [ ]:
if 'dbh' not in df.columns:
    print('DBH não disponível — biomassa não calculada.')
else:
    dbh_valid = df['dbh'].dropna()
    dbh_valid = dbh_valid[(dbh_valid > 0) & (dbh_valid < 500)]

    # Brown 1997
    ln_d = np.log(dbh_valid)
    agb_kg = np.exp(-2.289 + 2.649 * ln_d - 0.021 * ln_d**2)
    agb_mg = agb_kg / 1000  # Mg (toneladas) por árvore

    df.loc[dbh_valid.index, 'agb_mg'] = agb_mg

    print('=== Biomassa estimada (Brown 1997) ===')
    print(f'Total AGB:          {agb_mg.sum():.0f} Mg')
    print(f'AGB mediana/árvore: {agb_mg.median():.3f} Mg')
    print(f'AGB máx/árvore:     {agb_mg.max():.1f} Mg (DAP={dbh_valid[agb_mg.idxmax()]:.0f}cm)')

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].scatter(dbh_valid, agb_mg, s=1, alpha=0.3, color='forestgreen', rasterized=True)
    axes[0].set_xlabel('DAP (cm)'); axes[0].set_ylabel('AGB por árvore (Mg)')
    axes[0].set_title('Relação DAP × Biomassa (Brown 1997)')
    axes[0].set_xscale('log'); axes[0].set_yscale('log')

    if site_col and 'agb_mg' in df.columns:
        agb_by_site = df.groupby(site_col)['agb_mg'].sum().sort_values(ascending=False) / 1000  # Gg
        axes[1].bar(agb_by_site.index, agb_by_site.values, color='darkorange')
        axes[1].set_xlabel('Site'); axes[1].set_ylabel('AGB total (Gg)')
        axes[1].set_title('Biomassa total estimada por site')
        axes[1].tick_params(axis='x', rotation=45, labelsize=7)

    plt.tight_layout()
    plt.show()

## 7. Resumo por plot

Métricas agregadas no nível de parcela — a unidade de análise do modelo.

In [ ]:
if site_col and plot_col and 'dbh' in df.columns:
    agg = {
        'n_trees': ('dbh', 'count'),
        'dbh_mean': ('dbh', 'mean'),
        'dbh_max': ('dbh', 'max'),
        'dbh_std': ('dbh', 'std'),
    }
    if 'agb_mg' in df.columns:
        agg['agb_total_mg'] = ('agb_mg', 'sum')
    if 'scientific_name' in df.columns:
        agg['n_species'] = ('scientific_name', 'nunique')

    plot_df = df.groupby([site_col, plot_col]).agg(**agg).reset_index()
    print(f'{len(plot_df)} plots com dados')
    display(plot_df.describe().round(2))
else:
    print('Colunas de site/plot/dbh não disponíveis para agregação.')